# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [ ]:
import pandas as pd
import re

## Variable global
content = ""
sections_content = {}
data_report = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGTh_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [ ]:
## Global pattern
section_header_pattern = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)

## S1
s1_kv_pattern = re.compile(r"^([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ' ]+?)\s*:\s*(.*)$", re.MULTILINE)

##S2

## S6
s6_nucleide_line_pattern = re.compile(
    r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)",
    re.MULTILINE
)

## Extraction des header de chaque section
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [ ]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())
    data_report[section.strip()] = None

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport

In [ ]:
s1_key = sections_titles[0]
matches = s1_kv_pattern.findall(sections_content[s1_key])
data_report[s1_key] = {key.strip(): value.strip() for key, value in matches}
data_report[s1_key]

### S2 - RAPPORT ANALYSE DES PICS

### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

### S5 - RAPPORT LIMITES DE DETECTION

### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [ ]:
columns = [
    "Nucléide",
    "LD",
    "SD",
    "Limite Basse",
    "Limite Haute",
    "Moyenne Activité",
    "Moyenne Incertitude",
    "Meilleure Estimation Activité",
    "Estimation Incertitude"
]



lignes = re.findall(s6_nucleide_line_pattern, content)

data_report["RAPPORT LIMITES DE DETECTION  ISO 11929"] = pd.DataFrame(lignes, columns=columns)
df = data_report["RAPPORT LIMITES DE DETECTION  ISO 11929"]

df[columns[1:]] = df[columns[1:]].apply(pd.to_numeric, errors="coerce")
print(df)

df.to_csv("../data/rapport-RGTh_3cm_parsed.csv", index=False)


# Zone de Test

In [ ]:
sections_content